# STAM Optimizer Testing Phases

This notebook contains local and heavy benchmark phases for STAM, STAMLite, and requested baseline optimizers.

The heavy comparison set is fixed to: STAM Full, STAMLite, SGD + Momentum, RMSProp, Adagrad, NAdam, and LAMB.

All benchmark results are saved as JSON files under `results/notebook_runs/`.

## Phase Plan

- Phase 0: Environment, GPU, imports, and optimizer registry.
- Phase 1: Optimizer smoke tests and state-memory report.
- Phase 1b: Fair warmed timing with compile time separated from runtime.
- Phase 2: Synthetic convex regression.
- Phase 3: Synthetic non-stationary MLP classification.
- Phase 4: Small-batch stress test.
- Phase 5: Local real dataset if available.
- Phase 5b: Local advanced noisy non-stationary robustness.
- Phase 5c: Learning-rate robustness sweep.
- Phase 5d: Memory scaling.
- Phase 5e: STAM ablation.
- Phase 6: Heavy MNIST/Fashion-MNIST MLP and CNN benchmarks.
- Phase 7: Heavy CIFAR-10 CNN benchmark.
- Phase 8: Heavy synthetic tiny-transformer language-model benchmark.
- Phase 9: Heavy long-horizon non-stationary benchmark.
- Phase 10: Heavy optimizer hyperparameter sweep.
- Phase 11: Publishable aggregate report and ranking table.

Research framing: this notebook compares adaptive fidelity, compute, and memory. STAMLite is reported as a separate efficient proxy variant rather than an exact mathematical approximation of STAM Full. Heavy phases are enabled in this notebook configuration. They may download datasets and require substantial CPU/GPU time.

In [ ]:
# Phase 0: Environment setup
import os
import sys
import json
import time
from pathlib import Path
from dataclasses import asdict, dataclass

os.environ.setdefault('XLA_PYTHON_CLIENT_PREALLOCATE', 'false')

PROJECT_ROOT = Path('..').resolve() if Path.cwd().name == 'notebooks' else Path.cwd().resolve()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

RESULTS_DIR = PROJECT_ROOT / 'results' / 'notebook_runs'
RESULTS_DIR.mkdir(parents=True, exist_ok=True)

import numpy as np
import jax
import jax.numpy as jnp
from jax import random
import optax

from stam_optimizer import STAM, STAMLite

print('Project root:', PROJECT_ROOT)
print('JAX devices:', jax.devices())
print('Default backend:', jax.default_backend())
print('GPU available:', any(d.platform == 'gpu' for d in jax.devices()))

In [ ]:
# Global benchmark flags
RUN_LOCAL_LIGHT = True
RUN_OPTIONAL_REAL_DATASETS = True
RUN_MNIST = True
RUN_CIFAR10 = True
RUN_TINY_TRANSFORMER = True

LOCAL_SEEDS = [0, 1]
HEAVY_SEEDS = [0, 1, 2]

REQUIRED_HEAVY_OPTIMIZERS = [
    'stam_full',
    'stam_lite',
    'sgd_momentum',
    'rmsprop',
    'adagrad',
    'nadam',
    'lamb',
]

PUBLISHABLE_OPTIMIZERS = REQUIRED_HEAVY_OPTIMIZERS


def to_jsonable(x):
    if isinstance(x, (jnp.ndarray, np.ndarray)):
        return x.tolist()
    if isinstance(x, (np.float32, np.float64)):
        return float(x)
    if isinstance(x, (np.int32, np.int64)):
        return int(x)
    if isinstance(x, dict):
        return {k: to_jsonable(v) for k, v in x.items()}
    if isinstance(x, (list, tuple)):
        return [to_jsonable(v) for v in x]
    return x


def save_json(name, payload):
    path = RESULTS_DIR / name
    with open(path, 'w', encoding='utf-8') as f:
        json.dump(to_jsonable(payload), f, indent=2)
    print('Saved:', path)
    return path


def block_until_ready_tree(tree):
    for leaf in jax.tree_util.tree_leaves(tree):
        if hasattr(leaf, 'block_until_ready'):
            leaf.block_until_ready()
    return tree


def pytree_nbytes(tree):
    return int(sum(getattr(x, 'nbytes', 0) for x in jax.tree_util.tree_leaves(tree)))


RUN_HEAVY_REAL_DATASETS = True
RUN_HEAVY_IMAGE_BENCHMARKS = True
RUN_HEAVY_CIFAR10 = True
RUN_HEAVY_TINY_TRANSFORMER = True
RUN_HEAVY_LONG_HORIZON = True
RUN_HEAVY_HPARAM_SWEEP = True

HEAVY_CONFIG = {
    'image_train_limit': 12000,
    'image_test_limit': 2000,
    'image_steps': 300,
    'image_batch_size': 128,
    'cifar_train_limit': 20000,
    'cifar_test_limit': 5000,
    'cifar_steps': 600,
    'transformer_steps': 500,
    'long_horizon_steps': 300,
    'heavy_seeds': HEAVY_SEEDS,
}


def available_publishable_optimizers(require_all=True):
    registry = make_optimizer_registry()
    missing = [name for name in REQUIRED_HEAVY_OPTIMIZERS if name not in registry]
    if missing and require_all:
        raise RuntimeError(
            'Missing required heavy benchmark optimizers: '
            + ', '.join(missing)
            + '. Upgrade optax or install a version that provides these transforms.'
        )
    return [name for name in REQUIRED_HEAVY_OPTIMIZERS if name in registry]


In [ ]:
# Optimizer registry
class STAMFixed(STAM):
    def __init__(self, **kwargs):
        kwargs['adapt_strength'] = 0.0
        super().__init__(**kwargs)


class STAMConstBeta(STAM):
    def __init__(self, const_b1=0.81, **kwargs):
        kwargs['b1_base'] = const_b1
        kwargs['adapt_strength'] = 0.0
        super().__init__(**kwargs)


def make_optimizer_registry(learning_rate=1e-3, weight_decay=1e-2):
    registry = {
        'stam_full': ('subtract', lambda: STAM(learning_rate=learning_rate, weight_decay=weight_decay, adapt_strength=0.2)),
        'stam_lite': ('subtract', lambda: STAMLite(learning_rate=learning_rate, weight_decay=weight_decay, adapt_strength=0.2, beta1_update_interval=5, state_dtype=jnp.float32)),
        'stam_fixed': ('subtract', lambda: STAMFixed(learning_rate=learning_rate, weight_decay=weight_decay)),
        'stam_const_beta': ('subtract', lambda: STAMConstBeta(learning_rate=learning_rate, weight_decay=weight_decay)),
        'adam': ('optax', lambda: optax.adam(learning_rate=learning_rate)),
        'adamw': ('optax', lambda: optax.adamw(learning_rate=learning_rate, weight_decay=weight_decay)),
        'sgd': ('optax', lambda: optax.sgd(learning_rate=learning_rate)),
        'sgd_momentum': ('optax', lambda: optax.sgd(learning_rate=learning_rate, momentum=0.9)),
        'rmsprop': ('optax', lambda: optax.rmsprop(learning_rate=learning_rate)),
        'adagrad': ('optax', lambda: optax.adagrad(learning_rate=learning_rate)),
    }
    optional = {
        'nadam': lambda: optax.nadam(learning_rate=learning_rate),
        'radam': lambda: optax.radam(learning_rate=learning_rate),
        'lion': lambda: optax.lion(learning_rate=learning_rate, weight_decay=weight_decay),
        'lamb': lambda: optax.lamb(learning_rate=learning_rate, weight_decay=weight_decay),
        'adabelief': lambda: optax.adabelief(learning_rate=learning_rate),
    }
    for name, builder in optional.items():
        if hasattr(optax, name):
            registry[name] = ('optax', builder)
    return registry


OPTIMIZER_REGISTRY = make_optimizer_registry()
HEAVY_OPTIMIZERS = available_publishable_optimizers(require_all=True)
print('Optimizers:', sorted(OPTIMIZER_REGISTRY.keys()))
print('Required heavy comparison optimizers:', HEAVY_OPTIMIZERS)
save_json(
    'phase0_optimizer_registry.json',
    {
        'optimizers': sorted(OPTIMIZER_REGISTRY.keys()),
        'required_heavy_optimizers': REQUIRED_HEAVY_OPTIMIZERS,
        'heavy_optimizers': HEAVY_OPTIMIZERS,
        'devices': [str(d) for d in jax.devices()],
        'backend': jax.default_backend(),
        'stam_lite': {
            'variance_signal': 'EMA(mean(g^2)) - EMA(mean(g))^2',
            'beta1_update_interval': 5,
            'state_dtype': 'float32',
        },
    },
)


In [ ]:
# Shared training helpers
def apply_update(params, updates, kind):
    if kind == 'optax':
        return optax.apply_updates(params, updates)
    return jax.tree_util.tree_map(lambda p, u: p - u, params, updates)

def accuracy_from_logits(logits, y):
    return float(jnp.mean(jnp.argmax(logits, axis=-1) == y))

def summarize_runs(rows):
    summary = {}
    metric_keys = [
        'final_loss',
        'final_accuracy',
        'final_train_loss',
        'test_accuracy',
        'eval_loss',
        'loss_stability_std',
        'median_step_time',
        'seconds',
        'state_bytes',
    ]
    names = sorted(set(r['optimizer'] for r in rows if 'optimizer' in r))
    for name in names:
        group = [r for r in rows if r.get('optimizer') == name]
        for key in metric_keys:
            vals = [r[key] for r in group if key in r and r[key] is not None]
            if vals:
                summary.setdefault(name, {})[key + '_mean'] = float(np.mean(vals))
                summary.setdefault(name, {})[key + '_std'] = float(np.std(vals))
    return summary

## Phase 1: Smoke Tests and Memory

This phase checks optimizer initialization, one update step, finite updates, and optimizer-state memory.

It intentionally does not report runtime, because a one-step smoke test includes uneven JIT compilation and dispatch effects. Use the dedicated fair timing benchmark for speed comparisons.

In [ ]:
def run_phase1_smoke_and_memory():
    params = {'w': jnp.ones((128, 64)), 'b': jnp.ones((64,))}
    grads = {'w': jnp.ones((128, 64)) * 0.01, 'b': jnp.ones((64,)) * 0.01}
    rows = []
    for name, (kind, builder) in make_optimizer_registry().items():
        opt = builder()
        state = opt.init(params)
        updates, new_state = opt.update(grads, state, params)
        new_params = apply_update(params, updates, kind)
        block_until_ready_tree(new_params)
        rows.append({
            'optimizer': name,
            'kind': kind,
            'state_bytes': pytree_nbytes(state),
            'finite_update': bool(all(bool(jnp.all(jnp.isfinite(x))) for x in jax.tree_util.tree_leaves(updates))),
        })
    payload = {
        'phase': 'smoke_and_memory',
        'methodology_note': 'No runtime is reported here; one-step smoke timing includes uneven JIT compilation/dispatch artifacts. Use fair_timing.json for warmed blocked timings.',
        'rows': rows,
    }
    save_json('phase1_smoke_and_memory.json', payload)
    return payload

phase1_results = run_phase1_smoke_and_memory() if RUN_LOCAL_LIGHT else None
phase1_results

## Phase 1b: Fair Warmed Timing

Use this phase for runtime comparisons. It separates compile time from steady-state runtime, excludes warmup steps, JIT-compiles the full train step for every optimizer, and blocks after every measured step.

Do not use Phase 1 smoke output as speed evidence.

In [ ]:
# Optional fair timing command. Run manually when you want speed evidence.
# The output file is results/fair_timing.json.
FAIR_TIMING_COMMAND = (
    'python stam_optimizer/benchmarks/fair_timing.py '
    '--optimizers stam_full,stam_lite,sgd_momentum,rmsprop,adagrad,nadam,lamb '
    '--seeds 2 --warmup-steps 5 --timed-steps 20 '
    '--output results/fair_timing.json'
)
print(FAIR_TIMING_COMMAND)

## Phase 2: Synthetic Convex Regression

This phase trains a linear model on a small synthetic regression task.

In [ ]:
def make_regression(seed=0, n=512, d=32):
    key = random.PRNGKey(seed)
    k1, k2, k3 = random.split(key, 3)
    x = random.normal(k1, (n, d))
    true_w = random.normal(k2, (d, 1))
    y = x @ true_w + 0.05 * random.normal(k3, (n, 1))
    return x, y

def init_linear(seed, d):
    key = random.PRNGKey(seed)
    return {'w': random.normal(key, (d, 1)) * 0.01, 'b': jnp.zeros((1,))}

def linear_predict(params, x):
    return x @ params['w'] + params['b']

def mse_loss(params, x, y):
    return jnp.mean((linear_predict(params, x) - y) ** 2)

def train_regression_one(name, seed, steps=40, lr=1e-2):
    x, y = make_regression(seed)
    params = init_linear(seed + 100, x.shape[1])
    kind, builder = make_optimizer_registry(learning_rate=lr, weight_decay=1e-4)[name]
    opt = builder()
    state = opt.init(params)
    losses, times = [], []
    for _ in range(steps):
        start = time.time()
        loss, grads = jax.value_and_grad(mse_loss)(params, x, y)
        updates, state = opt.update(grads, state, params)
        params = apply_update(params, updates, kind)
        block_until_ready_tree((params, loss))
        losses.append(float(loss))
        times.append(time.time() - start)
    return {'optimizer': name, 'seed': seed, 'initial_loss': losses[0], 'final_loss': losses[-1], 'median_step_time': float(np.median(times[3:] if len(times) > 3 else times)), 'losses': losses}

def run_phase2_regression():
    rows = []
    for name in sorted(OPTIMIZER_REGISTRY.keys()):
        for seed in LOCAL_SEEDS:
            rows.append(train_regression_one(name, seed))
    payload = {'phase': 'synthetic_convex_regression', 'rows': rows, 'summary': summarize_runs(rows)}
    save_json('phase2_synthetic_regression.json', payload)
    return payload

phase2_results = run_phase2_regression() if RUN_LOCAL_LIGHT else None
phase2_results['summary'] if phase2_results else None

## Phase 3: Non-Stationary MLP Classification

This phase trains a small neural network on a synthetic shifted classification dataset.

In [ ]:
def make_shifted_classification(seed, n=768, d=64, classes=4):
    key = random.PRNGKey(seed)
    k1, k2, k3, k4 = random.split(key, 4)
    w1 = random.normal(k1, (d, classes))
    w2 = w1 + 0.6 * random.normal(k2, (d, classes))
    xa = random.normal(k3, (n // 2, d))
    xb = random.normal(k4, (n // 2, d)) + 0.7
    ya = jnp.argmax(xa @ w1, axis=-1)
    yb = jnp.argmax(xb @ w2, axis=-1)
    return jnp.concatenate([xa, xb]), jnp.concatenate([ya, yb])

def init_mlp(seed, sizes):
    keys = random.split(random.PRNGKey(seed), len(sizes) - 1)
    params = []
    for k, n_in, n_out in zip(keys, sizes[:-1], sizes[1:]):
        params.append({'w': random.normal(k, (n_in, n_out)) * jnp.sqrt(2.0 / n_in), 'b': jnp.zeros((n_out,))})
    return params

def mlp_forward(params, x):
    for layer in params[:-1]:
        x = jax.nn.relu(x @ layer['w'] + layer['b'])
    return x @ params[-1]['w'] + params[-1]['b']

def ce_loss(params, x, y):
    logits = mlp_forward(params, x)
    return -jnp.mean(jnp.sum(jax.nn.one_hot(y, logits.shape[-1]) * jax.nn.log_softmax(logits), axis=-1))

def train_mlp_one(name, seed, steps=30, lr=1e-3):
    x, y = make_shifted_classification(seed)
    params = init_mlp(seed + 1000, [64, 128, 64, 4])
    kind, builder = make_optimizer_registry(learning_rate=lr, weight_decay=1e-2)[name]
    opt = builder()
    state = opt.init(params)
    losses, accs, times = [], [], []
    for _ in range(steps):
        start = time.time()
        loss, grads = jax.value_and_grad(ce_loss)(params, x, y)
        updates, state = opt.update(grads, state, params)
        params = apply_update(params, updates, kind)
        block_until_ready_tree((params, loss))
        logits = mlp_forward(params, x)
        losses.append(float(loss))
        accs.append(accuracy_from_logits(logits, y))
        times.append(time.time() - start)
    return {'optimizer': name, 'seed': seed, 'final_loss': losses[-1], 'final_accuracy': accs[-1], 'median_step_time': float(np.median(times[3:] if len(times) > 3 else times)), 'losses': losses, 'accuracies': accs}

def run_phase3_mlp():
    rows = []
    for name in sorted(OPTIMIZER_REGISTRY.keys()):
        for seed in LOCAL_SEEDS:
            rows.append(train_mlp_one(name, seed))
    payload = {'phase': 'synthetic_nonstationary_mlp', 'rows': rows, 'summary': summarize_runs(rows)}
    save_json('phase3_nonstationary_mlp.json', payload)
    return payload

phase3_results = run_phase3_mlp() if RUN_LOCAL_LIGHT else None
phase3_results['summary'] if phase3_results else None

## Phase 4: Small-Batch Stress Test

This phase uses small batches to stress noisy gradient behavior.

In [ ]:
def batch_iter(seed, x, y, batch_size, steps):
    key = random.PRNGKey(seed)
    n = x.shape[0]
    for _ in range(steps):
        key, sub = random.split(key)
        idx = random.randint(sub, (batch_size,), 0, n)
        yield x[idx], y[idx]

def train_stress_one(name, seed, batch_size=8, steps=25, lr=1e-3):
    x, y = make_shifted_classification(seed, n=1024)
    params = init_mlp(seed + 2000, [64, 128, 64, 4])
    kind, builder = make_optimizer_registry(learning_rate=lr, weight_decay=1e-2)[name]
    opt = builder()
    state = opt.init(params)
    losses, accs, times = [], [], []
    for xb, yb in batch_iter(seed + 3000, x, y, batch_size, steps):
        start = time.time()
        loss, grads = jax.value_and_grad(ce_loss)(params, xb, yb)
        updates, state = opt.update(grads, state, params)
        params = apply_update(params, updates, kind)
        block_until_ready_tree((params, loss))
        losses.append(float(loss))
        accs.append(accuracy_from_logits(mlp_forward(params, x), y))
        times.append(time.time() - start)
    return {'optimizer': name, 'seed': seed, 'batch_size': batch_size, 'final_loss': losses[-1], 'final_accuracy': accs[-1], 'loss_stability_std': float(np.std(losses)), 'median_step_time': float(np.median(times[3:] if len(times) > 3 else times)), 'losses': losses, 'accuracies': accs}

def run_phase4_stress():
    rows = []
    for batch_size in [4, 8, 16]:
        for name in sorted(OPTIMIZER_REGISTRY.keys()):
            for seed in LOCAL_SEEDS:
                rows.append(train_stress_one(name, seed, batch_size=batch_size))
    payload = {'phase': 'small_batch_stress', 'rows': rows, 'summary': summarize_runs(rows)}
    save_json('phase4_small_batch_stress.json', payload)
    return payload

phase4_results = run_phase4_stress() if RUN_LOCAL_LIGHT else None
phase4_results['summary'] if phase4_results else None

## Phase 5: Local Real Dataset

This optional phase uses `sklearn.datasets.load_digits` if scikit-learn is installed.

In [ ]:
def run_phase5_digits_if_available():
    try:
        from sklearn.datasets import load_digits
        from sklearn.model_selection import train_test_split
    except Exception as exc:
        payload = {'phase': 'digits_dataset', 'status': 'skipped', 'reason': str(exc)}
        save_json('phase5_digits_skipped.json', payload)
        return payload
    data = load_digits()
    x = jnp.asarray(data.data, dtype=jnp.float32) / 16.0
    y = jnp.asarray(data.target, dtype=jnp.int32)
    rows = []
    for name in sorted(OPTIMIZER_REGISTRY.keys()):
        for seed in LOCAL_SEEDS:
            params = init_mlp(seed + 4000, [64, 128, 10])
            kind, builder = make_optimizer_registry(learning_rate=1e-3, weight_decay=1e-3)[name]
            opt = builder()
            state = opt.init(params)
            losses, accs, times = [], [], []
            for _ in range(50):
                start = time.time()
                loss, grads = jax.value_and_grad(ce_loss)(params, x, y)
                updates, state = opt.update(grads, state, params)
                params = apply_update(params, updates, kind)
                block_until_ready_tree((params, loss))
                losses.append(float(loss))
                accs.append(accuracy_from_logits(mlp_forward(params, x), y))
                times.append(time.time() - start)
            rows.append({'optimizer': name, 'seed': seed, 'final_loss': losses[-1], 'final_accuracy': accs[-1], 'median_step_time': float(np.median(times[3:])), 'losses': losses, 'accuracies': accs})
    payload = {'phase': 'digits_dataset', 'status': 'completed', 'rows': rows, 'summary': summarize_runs(rows)}
    save_json('phase5_digits_dataset.json', payload)
    return payload

phase5_results = run_phase5_digits_if_available() if RUN_OPTIONAL_REAL_DATASETS else None
phase5_results

## Phase 5b: Local Advanced Robustness

This local phase is designed to stress the claimed STAM regime: noisy, non-stationary gradients with abrupt distribution changes. It runs without internet and compares every optimizer in the registry, including STAM and STAMLite.

In [ ]:
def train_noisy_shift_one(name, seed, steps=60, batch_size=16, lr=1e-3, noise_scale=0.05):
    x, y = make_shifted_classification(seed, n=1536)
    params = init_mlp(seed + 5000, [64, 128, 64, 4])
    kind, builder = make_optimizer_registry(learning_rate=lr, weight_decay=1e-2)[name]
    opt = builder()
    state = opt.init(params)
    losses, accs = [], []
    key = random.PRNGKey(seed + 6000)
    for step, (xb, yb) in enumerate(batch_iter(seed + 7000, x, y, batch_size, steps)):
        if step == steps // 2:
            xb = xb + 1.25
            yb = (yb + 1) % 4
        loss, grads = jax.value_and_grad(ce_loss)(params, xb, yb)
        key, sub = random.split(key)
        leaves, treedef = jax.tree_util.tree_flatten(grads)
        noisy = [g + noise_scale * random.normal(random.fold_in(sub, i), g.shape) for i, g in enumerate(leaves)]
        grads = jax.tree_util.tree_unflatten(treedef, noisy)
        updates, state = opt.update(grads, state, params)
        params = apply_update(params, updates, kind)
        block_until_ready_tree((params, loss))
        losses.append(float(loss))
        accs.append(accuracy_from_logits(mlp_forward(params, x), y))
    return {
        'optimizer': name,
        'seed': seed,
        'final_loss': losses[-1],
        'final_accuracy': accs[-1],
        'recovery_loss_delta': losses[-1] - losses[steps // 2],
        'loss_stability_std': float(np.std(losses)),
        'losses': losses,
        'accuracies': accs,
    }

def run_phase5b_advanced_robustness():
    rows = []
    for name in sorted(OPTIMIZER_REGISTRY.keys()):
        for seed in LOCAL_SEEDS:
            rows.append(train_noisy_shift_one(name, seed))
    payload = {'phase': 'local_advanced_noisy_nonstationary_robustness', 'rows': rows, 'summary': summarize_runs(rows)}
    save_json('phase5b_advanced_robustness.json', payload)
    return payload

phase5b_results = run_phase5b_advanced_robustness() if RUN_LOCAL_LIGHT else None
phase5b_results['summary'] if phase5b_results else None

## Phase 5c: Learning-Rate Robustness Sweep

This phase tests whether optimizers remain finite and useful across several learning rates. It is local and intentionally small.

In [ ]:
def run_phase5c_lr_sweep():
    rows = []
    for lr in [3e-4, 1e-3, 3e-3, 1e-2]:
        for name in sorted(OPTIMIZER_REGISTRY.keys()):
            for seed in LOCAL_SEEDS:
                try:
                    row = train_mlp_one(name, seed, steps=25, lr=lr)
                    row['learning_rate'] = lr
                    row['finite'] = bool(np.isfinite(row['final_loss']))
                except Exception as exc:
                    row = {'optimizer': name, 'seed': seed, 'learning_rate': lr, 'finite': False, 'error': str(exc)}
                rows.append(row)
    payload = {'phase': 'learning_rate_robustness_sweep', 'rows': rows, 'summary': summarize_runs([r for r in rows if r.get('finite')])}
    save_json('phase5c_lr_sweep.json', payload)
    return payload

phase5c_results = run_phase5c_lr_sweep() if RUN_LOCAL_LIGHT else None
phase5c_results['summary'] if phase5c_results else None

## Phase 5d: Memory Scaling

This phase measures optimizer-state bytes over increasing parameter sizes. It is essential for validating the STAMLite memory-efficiency claim.

In [ ]:
def run_phase5d_memory_scaling():
    rows = []
    shapes = [(256, 256), (1024, 1024), (2048, 2048)]
    for shape in shapes:
        params = {'w': jnp.ones(shape), 'b': jnp.ones((shape[1],))}
        for name in sorted(OPTIMIZER_REGISTRY.keys()):
            kind, builder = make_optimizer_registry()[name]
            state = builder().init(params)
            rows.append({'optimizer': name, 'shape': list(shape), 'param_bytes': pytree_nbytes(params), 'state_bytes': pytree_nbytes(state)})
    payload = {'phase': 'optimizer_state_memory_scaling', 'rows': rows}
    save_json('phase5d_memory_scaling.json', payload)
    return payload

phase5d_results = run_phase5d_memory_scaling() if RUN_LOCAL_LIGHT else None
phase5d_results

## Phase 5e: STAM Ablation

This phase compares STAM Full, STAMLite, fixed-beta STAM, and constant low-beta STAM on the same non-stationary task. It isolates whether adaptive beta contributes beyond implementation details.

In [ ]:
def run_phase5e_stam_ablation():
    rows = []
    for name in ['stam_full', 'stam_lite', 'stam_fixed', 'stam_const_beta', 'adamw']:
        for seed in HEAVY_SEEDS:
            rows.append(train_noisy_shift_one(name, seed, steps=80, batch_size=16, lr=1e-3, noise_scale=0.05))
    payload = {'phase': 'stam_ablation_noisy_nonstationary', 'rows': rows, 'summary': summarize_runs(rows)}
    save_json('phase5e_stam_ablation.json', payload)
    return payload

phase5e_results = run_phase5e_stam_ablation() if RUN_LOCAL_LIGHT else None
phase5e_results['summary'] if phase5e_results else None

## Phase 6: MNIST/Fashion-MNIST MLP and CNN Benchmarks

This heavy phase uses real image datasets through `tensorflow.keras.datasets` when available. It may download data the first time. It compares STAM Full and STAMLite against SGD + Momentum, RMSProp, Adagrad, NAdam, and LAMB under the same training loop and exports JSON results.

In [ ]:
def load_keras_image_dataset(dataset_name, train_limit=None, test_limit=None):
    try:
        from tensorflow.keras.datasets import mnist, fashion_mnist, cifar10
    except Exception as exc:
        return None, {'status': 'skipped', 'reason': f'tensorflow.keras unavailable: {exc}'}
    loaders = {'mnist': mnist, 'fashion_mnist': fashion_mnist, 'cifar10': cifar10}
    if dataset_name not in loaders:
        return None, {'status': 'skipped', 'reason': f'unknown dataset: {dataset_name}'}
    try:
        (x_train, y_train), (x_test, y_test) = loaders[dataset_name].load_data()
    except Exception as exc:
        return None, {'status': 'skipped', 'reason': f'dataset load failed: {exc}'}
    if dataset_name in {'mnist', 'fashion_mnist'}:
        x_train = x_train[..., None]
        x_test = x_test[..., None]
    y_train = np.asarray(y_train).reshape(-1).astype(np.int32)
    y_test = np.asarray(y_test).reshape(-1).astype(np.int32)
    x_train = np.asarray(x_train, dtype=np.float32) / 255.0
    x_test = np.asarray(x_test, dtype=np.float32) / 255.0
    if train_limit:
        x_train, y_train = x_train[:train_limit], y_train[:train_limit]
    if test_limit:
        x_test, y_test = x_test[:test_limit], y_test[:test_limit]
    return (jnp.asarray(x_train), jnp.asarray(y_train), jnp.asarray(x_test), jnp.asarray(y_test)), {'status': 'loaded'}

def init_image_mlp(seed, input_shape, hidden=512, classes=10):
    key = random.PRNGKey(seed)
    k1, k2 = random.split(key)
    d = int(np.prod(input_shape))
    return {
        'w1': random.normal(k1, (d, hidden)) * jnp.sqrt(2.0 / d),
        'b1': jnp.zeros((hidden,)),
        'w2': random.normal(k2, (hidden, classes)) * jnp.sqrt(2.0 / hidden),
        'b2': jnp.zeros((classes,)),
    }

def image_mlp_forward(params, x):
    x = x.reshape((x.shape[0], -1))
    h = jax.nn.relu(x @ params['w1'] + params['b1'])
    return h @ params['w2'] + params['b2']

def init_cnn(seed, input_shape, classes=10, width=32):
    key = random.PRNGKey(seed)
    k1, k2, k3 = random.split(key, 3)
    channels = input_shape[-1]
    flat_features = (input_shape[0] // 4) * (input_shape[1] // 4) * (width * 2)
    return {
        'conv1_w': random.normal(k1, (3, 3, channels, width)) * jnp.sqrt(2.0 / (3 * 3 * channels)),
        'conv1_b': jnp.zeros((width,)),
        'conv2_w': random.normal(k2, (3, 3, width, width * 2)) * jnp.sqrt(2.0 / (3 * 3 * width)),
        'conv2_b': jnp.zeros((width * 2,)),
        'head_w': random.normal(k3, (flat_features, classes)) * jnp.sqrt(2.0 / flat_features),
        'head_b': jnp.zeros((classes,)),
    }

def max_pool_2x2(x):
    return jax.lax.reduce_window(x, -jnp.inf, jax.lax.max, (1, 2, 2, 1), (1, 2, 2, 1), 'VALID')

def cnn_forward(params, x):
    x = jax.lax.conv_general_dilated(x, params['conv1_w'], (1, 1), 'SAME', dimension_numbers=('NHWC', 'HWIO', 'NHWC')) + params['conv1_b']
    x = max_pool_2x2(jax.nn.relu(x))
    x = jax.lax.conv_general_dilated(x, params['conv2_w'], (1, 1), 'SAME', dimension_numbers=('NHWC', 'HWIO', 'NHWC')) + params['conv2_b']
    x = max_pool_2x2(jax.nn.relu(x))
    x = x.reshape((x.shape[0], -1))
    return x @ params['head_w'] + params['head_b']

def supervised_ce_loss(forward_fn, params, x, y):
    logits = forward_fn(params, x)
    return -jnp.mean(jnp.sum(jax.nn.one_hot(y, logits.shape[-1]) * jax.nn.log_softmax(logits), axis=-1))

def train_supervised_image_one(name, seed, dataset, model_name, steps, batch_size, lr, weight_decay):
    x_train, y_train, x_test, y_test = dataset
    input_shape = tuple(x_train.shape[1:])
    if model_name == 'mlp':
        params = init_image_mlp(seed + 8000, input_shape)
        forward_fn = image_mlp_forward
    else:
        params = init_cnn(seed + 8000, input_shape)
        forward_fn = cnn_forward
    kind, builder = make_optimizer_registry(learning_rate=lr, weight_decay=weight_decay)[name]
    opt = builder()
    state = opt.init(params)
    losses = []
    key = random.PRNGKey(seed + 9000)
    n = x_train.shape[0]
    start_time = time.time()
    for _ in range(steps):
        key, sub = random.split(key)
        idx = random.randint(sub, (batch_size,), 0, n)
        xb, yb = x_train[idx], y_train[idx]
        loss, grads = jax.value_and_grad(lambda p: supervised_ce_loss(forward_fn, p, xb, yb))(params)
        updates, state = opt.update(grads, state, params)
        params = apply_update(params, updates, kind)
        block_until_ready_tree((params, loss))
        losses.append(float(loss))
    logits = forward_fn(params, x_test)
    block_until_ready_tree(logits)
    return {
        'optimizer': name,
        'seed': seed,
        'model': model_name,
        'final_train_loss': losses[-1],
        'final_loss': losses[-1],
        'test_accuracy': accuracy_from_logits(logits, y_test),
        'final_accuracy': accuracy_from_logits(logits, y_test),
        'loss_stability_std': float(np.std(losses[-max(10, steps // 5):])),
        'seconds': time.time() - start_time,
        'steps': steps,
        'batch_size': batch_size,
        'learning_rate': lr,
    }

def run_phase6_mnist_fashion_benchmarks():
    rows = []
    for dataset_name in ['mnist', 'fashion_mnist']:
        dataset, status = load_keras_image_dataset(dataset_name, HEAVY_CONFIG['image_train_limit'], HEAVY_CONFIG['image_test_limit'])
        if dataset is None:
            rows.append({'dataset': dataset_name, **status})
            continue
        for model_name in ['mlp', 'cnn']:
            for name in available_publishable_optimizers():
                for seed in HEAVY_CONFIG['heavy_seeds']:
                    row = train_supervised_image_one(name, seed, dataset, model_name, HEAVY_CONFIG['image_steps'], HEAVY_CONFIG['image_batch_size'], 1e-3, 1e-4)
                    row['dataset'] = dataset_name
                    rows.append(row)
    completed = [r for r in rows if 'test_accuracy' in r]
    payload = {'phase': 'mnist_fashion_mnist_mlp_cnn', 'rows': rows, 'summary': summarize_runs(completed) if completed else {}}
    save_json('phase6_mnist_fashion_benchmarks.json', payload)
    return payload

phase6_results = run_phase6_mnist_fashion_benchmarks() if RUN_HEAVY_IMAGE_BENCHMARKS else None
phase6_results['summary'] if phase6_results else None

## Phase 7: CIFAR-10 CNN Benchmark

This heavy phase runs a larger real-image CNN benchmark on CIFAR-10. It compares STAM Full and STAMLite against SGD + Momentum, RMSProp, Adagrad, NAdam, and LAMB. It may download data and should preferably run on GPU for publishable timing.

In [ ]:
def run_phase7_cifar10_cnn_benchmark():
    dataset, status = load_keras_image_dataset('cifar10', HEAVY_CONFIG['cifar_train_limit'], HEAVY_CONFIG['cifar_test_limit'])
    if dataset is None:
        payload = {'phase': 'cifar10_cnn', **status}
        save_json('phase7_cifar10_cnn.json', payload)
        return payload
    rows = []
    for name in available_publishable_optimizers():
        for seed in HEAVY_CONFIG['heavy_seeds']:
            row = train_supervised_image_one(name, seed, dataset, 'cnn', HEAVY_CONFIG['cifar_steps'], HEAVY_CONFIG['image_batch_size'], 1e-3, 1e-4)
            row['dataset'] = 'cifar10'
            rows.append(row)
    payload = {'phase': 'cifar10_cnn', 'rows': rows, 'summary': summarize_runs(rows)}
    save_json('phase7_cifar10_cnn.json', payload)
    return payload

phase7_results = run_phase7_cifar10_cnn_benchmark() if RUN_HEAVY_CIFAR10 else None
phase7_results['summary'] if phase7_results and 'summary' in phase7_results else phase7_results

## Phase 8: Synthetic Tiny Transformer Language Model

This heavy phase uses a small causal transformer on synthetic token sequences. It needs no internet but is compute-heavy. It tests optimizer behavior on sequence-model dynamics rather than image/classification only, comparing STAM Full and STAMLite against SGD + Momentum, RMSProp, Adagrad, NAdam, and LAMB.

In [ ]:
def make_markov_tokens(seed, batch, seq_len, vocab_size):
    key = random.PRNGKey(seed)
    starts = random.randint(key, (batch, 1), 0, vocab_size)
    increments = random.bernoulli(random.fold_in(key, 1), 0.85, (batch, seq_len - 1)).astype(jnp.int32)
    noise = random.randint(random.fold_in(key, 2), (batch, seq_len - 1), 0, vocab_size)
    seq = [starts]
    cur = starts[:, 0]
    for i in range(seq_len - 1):
        cur = jnp.where(increments[:, i] == 1, (cur + 1) % vocab_size, noise[:, i])
        seq.append(cur[:, None])
    return jnp.concatenate(seq, axis=1)

def init_tiny_transformer(seed, vocab_size=64, seq_len=32, d_model=64, d_ff=128):
    keys = random.split(random.PRNGKey(seed), 10)
    scale = 1.0 / jnp.sqrt(d_model)
    return {
        'tok': random.normal(keys[0], (vocab_size, d_model)) * scale,
        'pos': random.normal(keys[1], (seq_len, d_model)) * scale,
        'wq': random.normal(keys[2], (d_model, d_model)) * scale,
        'wk': random.normal(keys[3], (d_model, d_model)) * scale,
        'wv': random.normal(keys[4], (d_model, d_model)) * scale,
        'wo': random.normal(keys[5], (d_model, d_model)) * scale,
        'ff1': random.normal(keys[6], (d_model, d_ff)) * scale,
        'ff1_b': jnp.zeros((d_ff,)),
        'ff2': random.normal(keys[7], (d_ff, d_model)) / jnp.sqrt(d_ff),
        'ff2_b': jnp.zeros((d_model,)),
        'head': random.normal(keys[8], (d_model, vocab_size)) * scale,
        'head_b': jnp.zeros((vocab_size,)),
    }

def tiny_transformer_forward(params, tokens):
    b, t = tokens.shape
    x = params['tok'][tokens] + params['pos'][:t]
    q = x @ params['wq']
    k = x @ params['wk']
    v = x @ params['wv']
    att = (q @ jnp.swapaxes(k, -1, -2)) / jnp.sqrt(q.shape[-1])
    mask = jnp.tril(jnp.ones((t, t), dtype=bool))
    att = jnp.where(mask[None, :, :], att, -1e9)
    x = x + (jax.nn.softmax(att, axis=-1) @ v) @ params['wo']
    x = x + (jax.nn.gelu(x @ params['ff1'] + params['ff1_b']) @ params['ff2'] + params['ff2_b'])
    return x @ params['head'] + params['head_b']

def transformer_lm_loss(params, tokens):
    logits = tiny_transformer_forward(params, tokens[:, :-1])
    targets = tokens[:, 1:]
    return -jnp.mean(jnp.sum(jax.nn.one_hot(targets, logits.shape[-1]) * jax.nn.log_softmax(logits), axis=-1))

def train_tiny_transformer_one(name, seed, steps=500, batch_size=64, seq_len=32, vocab_size=64, lr=3e-4):
    params = init_tiny_transformer(seed + 10000, vocab_size=vocab_size, seq_len=seq_len)
    kind, builder = make_optimizer_registry(learning_rate=lr, weight_decay=1e-2)[name]
    opt = builder()
    state = opt.init(params)
    losses = []
    start_time = time.time()
    for step in range(steps):
        tokens = make_markov_tokens(seed + 11000 + step, batch_size, seq_len, vocab_size)
        loss, grads = jax.value_and_grad(transformer_lm_loss)(params, tokens)
        updates, state = opt.update(grads, state, params)
        params = apply_update(params, updates, kind)
        block_until_ready_tree((params, loss))
        losses.append(float(loss))
    eval_tokens = make_markov_tokens(seed + 12000, batch_size * 2, seq_len, vocab_size)
    eval_loss = float(transformer_lm_loss(params, eval_tokens))
    return {'optimizer': name, 'seed': seed, 'final_train_loss': losses[-1], 'final_loss': losses[-1], 'eval_loss': eval_loss, 'loss_stability_std': float(np.std(losses[-max(10, steps // 5):])), 'seconds': time.time() - start_time, 'steps': steps}

def run_phase8_tiny_transformer():
    rows = []
    for name in available_publishable_optimizers():
        for seed in HEAVY_CONFIG['heavy_seeds']:
            rows.append(train_tiny_transformer_one(name, seed, steps=HEAVY_CONFIG['transformer_steps']))
    payload = {'phase': 'synthetic_tiny_transformer_lm', 'rows': rows, 'summary': summarize_runs(rows)}
    save_json('phase8_tiny_transformer.json', payload)
    return payload

phase8_results = run_phase8_tiny_transformer() if RUN_HEAVY_TINY_TRANSFORMER else None
phase8_results['summary'] if phase8_results else None

## Phase 9: Long-Horizon Non-Stationary Training

This heavy phase extends the synthetic non-stationary task to longer training and larger models. It is useful for checking whether early advantages persist or vanish across STAM Full, STAMLite, SGD + Momentum, RMSProp, Adagrad, NAdam, and LAMB.

In [ ]:
def train_long_horizon_one(name, seed, steps=300, lr=1e-3):
    x, y = make_shifted_classification(seed, n=4096, d=128, classes=8)
    params = init_mlp(seed + 13000, [128, 512, 256, 128, 8])
    kind, builder = make_optimizer_registry(learning_rate=lr, weight_decay=1e-2)[name]
    opt = builder()
    state = opt.init(params)
    losses, accs = [], []
    for step, (xb, yb) in enumerate(batch_iter(seed + 14000, x, y, 64, steps)):
        if step in {steps // 3, 2 * steps // 3}:
            xb = xb + 0.75
        loss, grads = jax.value_and_grad(ce_loss)(params, xb, yb)
        updates, state = opt.update(grads, state, params)
        params = apply_update(params, updates, kind)
        block_until_ready_tree((params, loss))
        losses.append(float(loss))
        if step % 10 == 0 or step == steps - 1:
            accs.append(accuracy_from_logits(mlp_forward(params, x), y))
    return {'optimizer': name, 'seed': seed, 'final_loss': losses[-1], 'final_accuracy': accs[-1], 'loss_stability_std': float(np.std(losses[-50:])), 'losses': losses, 'sampled_accuracies': accs}

def run_phase9_long_horizon():
    rows = []
    for name in available_publishable_optimizers():
        for seed in HEAVY_CONFIG['heavy_seeds']:
            rows.append(train_long_horizon_one(name, seed, steps=HEAVY_CONFIG['long_horizon_steps']))
    payload = {'phase': 'long_horizon_nonstationary_mlp', 'rows': rows, 'summary': summarize_runs(rows)}
    save_json('phase9_long_horizon.json', payload)
    return payload

phase9_results = run_phase9_long_horizon() if RUN_HEAVY_LONG_HORIZON else None
phase9_results['summary'] if phase9_results else None

## Phase 10: Optimizer Hyperparameter Sweep

This heavy phase performs a fairer robustness comparison by sweeping learning rates and weight decay on the same non-stationary task. Publication claims should use tuned or swept baselines, not one default setting.

In [ ]:
def run_phase10_hparam_sweep():
    rows = []
    lrs = [1e-4, 3e-4, 1e-3, 3e-3]
    weight_decays = [0.0, 1e-4, 1e-2]
    for name in available_publishable_optimizers():
        for lr in lrs:
            for wd in weight_decays:
                for seed in HEAVY_CONFIG['heavy_seeds']:
                    try:
                        x, y = make_shifted_classification(seed, n=1536)
                        params = init_mlp(seed + 15000, [64, 128, 64, 4])
                        kind, builder = make_optimizer_registry(learning_rate=lr, weight_decay=wd)[name]
                        opt = builder()
                        state = opt.init(params)
                        for xb, yb in batch_iter(seed + 16000, x, y, 32, 80):
                            loss, grads = jax.value_and_grad(ce_loss)(params, xb, yb)
                            updates, state = opt.update(grads, state, params)
                            params = apply_update(params, updates, kind)
                        logits = mlp_forward(params, x)
                        row = {
                            'optimizer': name,
                            'seed': seed,
                            'learning_rate': lr,
                            'weight_decay': wd,
                            'final_loss': float(ce_loss(params, x, y)),
                            'final_accuracy': accuracy_from_logits(logits, y),
                            'finite': True,
                        }
                    except Exception as exc:
                        row = {
                            'optimizer': name,
                            'seed': seed,
                            'learning_rate': lr,
                            'weight_decay': wd,
                            'finite': False,
                            'error': str(exc),
                        }
                    rows.append(row)
    completed = [r for r in rows if r.get('finite')]
    payload = {'phase': 'optimizer_hparam_sweep', 'rows': rows, 'summary': summarize_runs(completed)}
    save_json('phase10_hparam_sweep.json', payload)
    return payload


phase10_results = run_phase10_hparam_sweep() if RUN_HEAVY_HPARAM_SWEEP else None
phase10_results['summary'] if phase10_results else None

## Phase 11: Publishable Aggregate Report

This phase aggregates all available JSON benchmark outputs into a single report. It does not run training and is safe to execute locally.

In [ ]:
def ci95(values):
    values = np.asarray(values, dtype=np.float64)
    values = values[np.isfinite(values)]
    if len(values) == 0:
        return None
    if len(values) == 1:
        return 0.0
    return float(1.96 * values.std(ddof=1) / np.sqrt(len(values)))

def aggregate_benchmark_files():
    aggregate = {'files': {}, 'optimizer_metrics': {}}
    for path in sorted(RESULTS_DIR.glob('phase*.json')):
        try:
            payload = json.loads(path.read_text(encoding='utf-8'))
        except Exception:
            continue
        rows = payload.get('rows', [])
        aggregate['files'][path.name] = {'phase': payload.get('phase'), 'rows': len(rows)}
        for row in rows:
            opt = row.get('optimizer')
            if not opt:
                continue
            bucket = aggregate['optimizer_metrics'].setdefault(opt, {'final_loss': [], 'final_accuracy': [], 'test_accuracy': [], 'eval_loss': [], 'state_bytes': []})
            for key in list(bucket.keys()):
                if key in row and row[key] is not None:
                    bucket[key].append(row[key])
    for opt, metrics in aggregate['optimizer_metrics'].items():
        for key, values in list(metrics.items()):
            if values:
                arr = np.asarray(values, dtype=np.float64)
                metrics[key] = {'n': int(len(arr)), 'mean': float(np.mean(arr)), 'std': float(np.std(arr)), 'ci95': ci95(arr.tolist())}
            else:
                metrics[key] = {'n': 0}
    save_json('phase11_publishable_aggregate_report.json', aggregate)
    return aggregate

phase11_results = aggregate_benchmark_files()
phase11_results

## Final Notes

Heavy phases compare STAM Full and STAMLite against SGD + Momentum, RMSProp, Adagrad, NAdam, and LAMB. Use a GPU machine for publishable timing. Do not claim superiority from synthetic results alone; real-data phases and multi-seed confidence intervals should drive any final claims.